In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Install required packages
!pip install -q indic-transliteration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 4.2 MB/s eta 0:00:00


In [3]:
# ================================
# IMPORT REQUIRED LIBRARIES
# ================================

import nltk
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter

# NLTK
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import indian

# Download required NLTK resources
nltk.download('indian')
nltk.download('punkt')
nltk.download('punkt_tab')

print("Libraries imported successfully!")

Libraries imported successfully!


[nltk_data] Downloading package indian to /usr/share/nltk_data...
[nltk_data]   Package indian is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# ============================================
# Q1(a): EXPLORE NLTK INDIAN CORPUS
# ============================================

print("Files available in NLTK Indian Corpus:")
print(indian.fileids())

Files available in NLTK Indian Corpus:
['bangla.pos', 'hindi.pos', 'marathi.pos', 'telugu.pos']


In [5]:
# ============================================
# LOAD HINDI TAGGED DATA
# ============================================

# Find a Hindi file automatically
hindi_files = [f for f in indian.fileids() if 'hindi' in f.lower()]

print("Hindi files found:")
print(hindi_files)

# Use the first Hindi file
hindi_file = hindi_files[0]

# Load tagged sentences
hindi_tagged = indian.tagged_sents(hindi_file)

print("Number of sentences:", len(hindi_tagged))
print("\nFirst 3 sentences:")
for sentence in hindi_tagged[:3]:
    print(sentence)

Hindi files found:
['hindi.pos']
Number of sentences: 540

First 3 sentences:
[('पूर्ण', 'JJ'), ('प्रतिबंध', 'NN'), ('हटाओ', 'VFM'), (':', 'SYM'), ('इराक', 'NNP')]
[('संयुक्त', 'NNC'), ('राष्ट्र', 'NN'), ('।', 'SYM')]
[('इराक', 'NNP'), ('के', 'PREP'), ('विदेश', 'NNC'), ('मंत्री', 'NN'), ('ने', 'PREP'), ('अमरीका', 'NNP'), ('के', 'PREP'), ('उस', 'PRP'), ('प्रस्ताव', 'NN'), ('का', 'PREP'), ('मजाक', 'NVB'), ('उड़ाया', 'VFM'), ('है', 'VAUX'), (',', 'PUNC'), ('जिसमें', 'PRP'), ('अमरीका', 'NNP'), ('ने', 'PREP'), ('संयुक्त', 'NNC'), ('राष्ट्र', 'NN'), ('के', 'PREP'), ('प्रतिबंधों', 'NN'), ('को', 'PREP'), ('इराकी', 'JJ'), ('नागरिकों', 'NN'), ('के', 'PREP'), ('लिए', 'PREP'), ('कम', 'INTF'), ('हानिकारक', 'JJ'), ('बनाने', 'VNN'), ('के', 'PREP'), ('लिए', 'PREP'), ('कहा', 'VFM'), ('है', 'VAUX'), ('।', 'PUNC')]


In [6]:
# ============================================
# Q1(b): EXTRACT AT LEAST 500 SENTENCES
# ============================================

# Extract up to 500 sentences
num_sentences = min(500, len(hindi_tagged))

hindi_sentences = hindi_tagged[:num_sentences]

print("Number of sentences extracted:", len(hindi_sentences))

# Convert tagged sentences into plain text
plain_hindi_sentences = []

for sentence in hindi_sentences:
    text = " ".join([word for word, tag in sentence])
    plain_hindi_sentences.append(text)

print("\nSample sentences:")
for i, sentence in enumerate(plain_hindi_sentences[:5], 1):
    print(f"{i}. {sentence}")

Number of sentences extracted: 500

Sample sentences:
1. पूर्ण प्रतिबंध हटाओ : इराक
2. संयुक्त राष्ट्र ।
3. इराक के विदेश मंत्री ने अमरीका के उस प्रस्ताव का मजाक उड़ाया है , जिसमें अमरीका ने संयुक्त राष्ट्र के प्रतिबंधों को इराकी नागरिकों के लिए कम हानिकारक बनाने के लिए कहा है ।
4. विदेश मंत्री का कहना है कि चूंकि बगदाद संयुक्त राष्ट्र की मांगों का पालन करते हुए अपने भारी विनाशकारी हथियारों को नष्ट कर रहा है ।
5. लिहाजा प्रतिबंधों को पूर्ण रूप से उठा दिया जाना चाहिए ।


In [7]:
# ============================================
# Q1(c): TOKENIZATION
# ============================================

all_tokens = []

for sentence in plain_hindi_sentences:
    tokens = wordpunct_tokenize(sentence)
    all_tokens.extend(tokens)

print("First 50 tokens:")
print(all_tokens[:50])

print("\nTotal raw tokens:", len(all_tokens))

First 50 tokens:
['प', 'ू', 'र', '्', 'ण', 'प', '्', 'रत', 'ि', 'ब', 'ं', 'ध', 'हट', 'ा', 'ओ', ':', 'इर', 'ा', 'क', 'स', 'ं', 'य', 'ु', 'क', '्', 'त', 'र', 'ा', 'ष', '्', 'ट', '्', 'र', '।', 'इर', 'ा', 'क', 'क', 'े', 'व', 'ि', 'द', 'े', 'श', 'म', 'ं', 'त', '्', 'र', 'ी']

Total raw tokens: 27471


In [8]:
# ============================================
# BASIC PREPROCESSING
# ============================================

def preprocess_indic_tokens(tokens):
    cleaned = []

    for token in tokens:
        # Unicode normalization
        token = unicodedata.normalize("NFC", token)

        # Remove extra whitespace
        token = token.strip()

        # Keep meaningful tokens
        if token and not token.isspace():
            cleaned.append(token)

    return cleaned


clean_tokens = preprocess_indic_tokens(all_tokens)

print("Tokens after preprocessing:", len(clean_tokens))
print("\nSample cleaned tokens:")
print(clean_tokens[:50])

Tokens after preprocessing: 27471

Sample cleaned tokens:
['प', 'ू', 'र', '्', 'ण', 'प', '्', 'रत', 'ि', 'ब', 'ं', 'ध', 'हट', 'ा', 'ओ', ':', 'इर', 'ा', 'क', 'स', 'ं', 'य', 'ु', 'क', '्', 'त', 'र', 'ा', 'ष', '्', 'ट', '्', 'र', '।', 'इर', 'ा', 'क', 'क', 'े', 'व', 'ि', 'द', 'े', 'श', 'म', 'ं', 'त', '्', 'र', 'ी']


In [9]:
# ============================================
# Q1(d): TOTAL TOKENS AND VOCABULARY SIZE
# ============================================

total_tokens = len(clean_tokens)
vocabulary = set(clean_tokens)
vocabulary_size = len(vocabulary)

print("Corpus Statistics")
print("=" * 40)
print("Number of sentences :", len(plain_hindi_sentences))
print("Total tokens        :", total_tokens)
print("Vocabulary size     :", vocabulary_size)

Corpus Statistics
Number of sentences : 500
Total tokens        : 27471
Vocabulary size     : 831


In [10]:
# ============================================
# Q1(e): TOP 10 MOST FREQUENT TOKENS
# ============================================

token_frequency = Counter(clean_tokens)

top_10 = token_frequency.most_common(10)

print("Top 10 Most Frequent Tokens")
print("=" * 40)

for token, frequency in top_10:
    print(f"{token:<20} {frequency}")

Top 10 Most Frequent Tokens
ा                    2711
े                    1878
क                    1616
्                    1601
ि                    1314
ी                    1193
र                    1004
न                    794
म                    714
स                    691


In [11]:
# Create a clean table for the report

stats_df = pd.DataFrame({
    "Metric": [
        "Number of Sentences",
        "Total Tokens",
        "Vocabulary Size"
    ],
    "Value": [
        len(plain_hindi_sentences),
        total_tokens,
        vocabulary_size
    ]
})

stats_df

,Metric,Value
0,Number of Sentences,500
1,Total Tokens,27471
2,Vocabulary Size,831


In [12]:
# ============================================
# Q2(a): CREATE 20 NOISY INDIC TEXT SAMPLES
# ============================================

noisy_texts = [
    "नमस्ते    दुनिया!!!",
    "आज मौसम बहुत अच्छा है!!!",
    "मुझे   NLP सीखना है....",
    "यह   एक बहुत अच्छा   प्रोजेक्ट है",
    "आप कैसे हैं????",
    "मुझे Python बहुत पसंद है!!!",
    "कल कॉलेज जाना है......",
    "यह बहुत अच्छा है!!!!!!",
    "नमस्ते     दोस्त",
    "आज १० बजे मीटिंग है",
    "मेरा नंबर ९८७६५४३२१० है",
    "HELLO   DOST!!!",
    "NLP IS AMAZING!!!",
    "hello     duniya",
    "Python बहुत EASY है!!!",
    "क्या आप READY हैं???",
    "मुझे AI में INTEREST है!!!",
    "बहुत बढ़िया!!!!!!",
    "आज का दिन बहुत अच्छा है :)",
    "वाहhhhhhh!!! यह कमाल है!!!!"
]

print("Number of noisy samples:", len(noisy_texts))

for i, text in enumerate(noisy_texts, 1):
    print(f"{i}. {text}")

Number of noisy samples: 20
1. नमस्ते    दुनिया!!!
2. आज मौसम बहुत अच्छा है!!!
3. मुझे   NLP सीखना है....
4. यह   एक बहुत अच्छा   प्रोजेक्ट है
5. आप कैसे हैं????
6. मुझे Python बहुत पसंद है!!!
7. कल कॉलेज जाना है......
8. यह बहुत अच्छा है!!!!!!
9. नमस्ते     दोस्त
10. आज १० बजे मीटिंग है
11. मेरा नंबर ९८७६५४३२१० है
12. HELLO   DOST!!!
13. NLP IS AMAZING!!!
14. hello     duniya
15. Python बहुत EASY है!!!
16. क्या आप READY हैं???
17. मुझे AI में INTEREST है!!!
18. बहुत बढ़िया!!!!!!
19. आज का दिन बहुत अच्छा है :)
20. वाहhhhhhh!!! यह कमाल है!!!!


In [13]:
# ============================================
# Q2(b,c,d): NORMALIZATION FUNCTIONS
# ============================================

# Mapping Indic digits to English digits
indic_digit_map = str.maketrans(
    "०१२३४५६७८९",
    "0123456789"
)


def normalize_indic_text(text):
    """
    Normalize Indic / mixed-language text.

    Steps:
    1. Unicode NFC normalization
    2. Normalize Indic digits to English digits
    3. Normalize repeated punctuation
    4. Remove repeated characters
    5. Normalize whitespace
    6. Lowercase Roman-script words
    """

    # Step 1: Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Step 2: Convert Indic digits
    text = text.translate(indic_digit_map)

    # Step 3: Normalize repeated punctuation
    text = re.sub(r'([!?.,])\1+', r'\1', text)

    # Step 4: Reduce excessive repeated Roman characters
    # Example: hellooooo -> helloo
    text = re.sub(r'([A-Za-z])\1{2,}', r'\1\1', text)

    # Step 5: Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 6: Lowercase Roman-script portions
    text = text.lower()

    return text


normalized_texts = [
    normalize_indic_text(text)
    for text in noisy_texts
]

In [14]:
# ============================================
# Q2(e): ORIGINAL VS NORMALIZED TEXT
# ============================================

normalization_df = pd.DataFrame({
    "Original Text": noisy_texts,
    "Normalized Text": normalized_texts
})

pd.set_option("display.max_colwidth", 100)

normalization_df

,Original Text,Normalized Text
0,नमस्ते दुनिया!!!,नमस्ते दुनिया!
1,आज मौसम बहुत अच्छा है!!!,आज मौसम बहुत अच्छा है!
2,मुझे NLP सीखना है....,मुझे nlp सीखना है.
3,यह एक बहुत अच्छा प्रोजेक्ट है,यह एक बहुत अच्छा प्रोजेक्ट है
4,आप कैसे हैं????,आप कैसे हैं?
5,मुझे Python बहुत पसंद है!!!,मुझे python बहुत पसंद है!
6,कल कॉलेज जाना है......,कल कॉलेज जाना है.
7,यह बहुत अच्छा है!!!!!!,यह बहुत अच्छा है!
8,नमस्ते दोस्त,नमस्ते दोस्त
9,आज १० बजे मीटिंग है,आज 10 बजे मीटिंग है


In [15]:
# ============================================
# DISPLAY CHANGES CLEARLY
# ============================================

for i, (original, normalized) in enumerate(
    zip(noisy_texts, normalized_texts), 1
):
    print(f"\nSample {i}")
    print("Original   :", original)
    print("Normalized :", normalized)


Sample 1
Original   : नमस्ते    दुनिया!!!
Normalized : नमस्ते दुनिया!

Sample 2
Original   : आज मौसम बहुत अच्छा है!!!
Normalized : आज मौसम बहुत अच्छा है!

Sample 3
Original   : मुझे   NLP सीखना है....
Normalized : मुझे nlp सीखना है.

Sample 4
Original   : यह   एक बहुत अच्छा   प्रोजेक्ट है
Normalized : यह एक बहुत अच्छा प्रोजेक्ट है

Sample 5
Original   : आप कैसे हैं????
Normalized : आप कैसे हैं?

Sample 6
Original   : मुझे Python बहुत पसंद है!!!
Normalized : मुझे python बहुत पसंद है!

Sample 7
Original   : कल कॉलेज जाना है......
Normalized : कल कॉलेज जाना है.

Sample 8
Original   : यह बहुत अच्छा है!!!!!!
Normalized : यह बहुत अच्छा है!

Sample 9
Original   : नमस्ते     दोस्त
Normalized : नमस्ते दोस्त

Sample 10
Original   : आज १० बजे मीटिंग है
Normalized : आज 10 बजे मीटिंग है

Sample 11
Original   : मेरा नंबर ९८७६५४३२१० है
Normalized : मेरा नंबर 9876543210 है

Sample 12
Original   : HELLO   DOST!!!
Normalized : hello dost!

Sample 13
Original   : NLP IS AMAZING!!!
Normalized : nlp is am

In [16]:
# ============================================
# Q3: TRANSLITERATION
# ============================================

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

print("Transliteration library loaded successfully.")

Transliteration library loaded successfully.


In [17]:
# ============================================
# Q3(a): TWO INDIC SCRIPTS
# ============================================

hindi_samples = [
    "नमस्ते",
    "मेरा नाम राहुल है",
    "भारत एक सुंदर देश है",
    "मुझे हिंदी भाषा पसंद है",
    "आज मौसम अच्छा है"
]

bengali_samples = [
    "নমস্কার",
    "আমার নাম রাহুল",
    "ভারত একটি সুন্দর দেশ",
    "আমি বাংলা ভাষা পছন্দ করি",
    "আজ আবহাওয়া ভালো"
]

print("Hindi Samples:")
for text in hindi_samples:
    print(text)

print("\nBengali Samples:")
for text in bengali_samples:
    print(text)

Hindi Samples:
नमस्ते
मेरा नाम राहुल है
भारत एक सुंदर देश है
मुझे हिंदी भाषा पसंद है
आज मौसम अच्छा है

Bengali Samples:
নমস্কার
আমার নাম রাহুল
ভারত একটি সুন্দর দেশ
আমি বাংলা ভাষা পছন্দ করি
আজ আবহাওয়া ভালো


In [18]:
# ============================================
# Q3(b): UNICODE ANALYSIS
# ============================================

def unicode_info(text):
    result = []

    for char in text:
        if not char.isspace():
            result.append({
                "Character": char,
                "Unicode": f"U+{ord(char):04X}",
                "Unicode Name": unicodedata.name(char, "UNKNOWN")
            })

    return pd.DataFrame(result)


print("Unicode information for Hindi word:")
display(unicode_info("नमस्ते"))

print("\nUnicode information for Bengali word:")
display(unicode_info("নমস্কার"))

Unicode information for Hindi word:


,Character,Unicode,Unicode Name
0,न,U+0928,DEVANAGARI LETTER NA
1,म,U+092E,DEVANAGARI LETTER MA
2,स,U+0938,DEVANAGARI LETTER SA
3,्,U+094D,DEVANAGARI SIGN VIRAMA
4,त,U+0924,DEVANAGARI LETTER TA
5,े,U+0947,DEVANAGARI VOWEL SIGN E



Unicode information for Bengali word:


,Character,Unicode,Unicode Name
0,ন,U+09A8,BENGALI LETTER NA
1,ম,U+09AE,BENGALI LETTER MA
2,স,U+09B8,BENGALI LETTER SA
3,্,U+09CD,BENGALI SIGN VIRAMA
4,ক,U+0995,BENGALI LETTER KA
5,া,U+09BE,BENGALI VOWEL SIGN AA
6,র,U+09B0,BENGALI LETTER RA


In [19]:
# ============================================
# Q3(c): UNICODE NORMALIZATION
# ============================================

normalized_hindi = [
    unicodedata.normalize("NFC", text)
    for text in hindi_samples
]

normalized_bengali = [
    unicodedata.normalize("NFC", text)
    for text in bengali_samples
]

print("Normalized Hindi:")
for text in normalized_hindi:
    print(text)

print("\nNormalized Bengali:")
for text in normalized_bengali:
    print(text)

Normalized Hindi:
नमस्ते
मेरा नाम राहुल है
भारत एक सुंदर देश है
मुझे हिंदी भाषा पसंद है
आज मौसम अच्छा है

Normalized Bengali:
নমস্কার
আমার নাম রাহুল
ভারত একটি সুন্দর দেশ
আমি বাংলা ভাষা পছন্দ করি
আজ আবহাওয়া ভালো


In [20]:
# ============================================
# Q3(d): HINDI -> ROMAN
# ============================================

hindi_roman = [
    transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    for text in normalized_hindi
]

print("Hindi Transliteration")
print("=" * 50)

for original, converted in zip(normalized_hindi, hindi_roman):
    print(f"{original:<30} -> {converted}")

Hindi Transliteration
नमस्ते                         -> namaste
मेरा नाम राहुल है              -> merA nAma rAhula hai
भारत एक सुंदर देश है           -> bhArata eka suMdara desha hai
मुझे हिंदी भाषा पसंद है        -> mujhe hiMdI bhAShA pasaMda hai
आज मौसम अच्छा है               -> Aja mausama achChA hai


In [21]:
# ============================================
# Q3(d): BENGALI -> ROMAN
# ============================================

bengali_roman = [
    transliterate(text, sanscript.BENGALI, sanscript.ITRANS)
    for text in normalized_bengali
]

print("Bengali Transliteration")
print("=" * 50)

for original, converted in zip(normalized_bengali, bengali_roman):
    print(f"{original:<30} -> {converted}")

Bengali Transliteration
নমস্কার                        -> namaskAra
আমার নাম রাহুল                 -> AmAra nAma rAhula
ভারত একটি সুন্দর দেশ           -> bhArata ekaTi sundara desha
আমি বাংলা ভাষা পছন্দ করি       -> Ami vAMlA bhAShA paChanda kari
আজ আবহাওয়া ভালো               -> Aja AvahAoya়A bhAlo


In [22]:
# ============================================
# Q3(e): ORIGINAL, NORMALIZED AND CONVERTED
# ============================================

script_comparison = pd.DataFrame({
    "Language": ["Hindi"] * 5 + ["Bengali"] * 5,
    "Original": hindi_samples + bengali_samples,
    "Normalized": normalized_hindi + normalized_bengali,
    "Roman / ITRANS": hindi_roman + bengali_roman
})

script_comparison

,Language,Original,Normalized,Roman / ITRANS
0,Hindi,नमस्ते,नमस्ते,namaste
1,Hindi,मेरा नाम राहुल है,मेरा नाम राहुल है,merA nAma rAhula hai
2,Hindi,भारत एक सुंदर देश है,भारत एक सुंदर देश है,bhArata eka suMdara desha hai
3,Hindi,मुझे हिंदी भाषा पसंद है,मुझे हिंदी भाषा पसंद है,mujhe hiMdI bhAShA pasaMda hai
4,Hindi,आज मौसम अच्छा है,आज मौसम अच्छा है,Aja mausama achChA hai
5,Bengali,নমস্কার,নমস্কার,namaskAra
6,Bengali,আমার নাম রাহুল,আমার নাম রাহুল,AmAra nAma rAhula
7,Bengali,ভারত একটি সুন্দর দেশ,ভারত একটি সুন্দর দেশ,bhArata ekaTi sundara desha
8,Bengali,আমি বাংলা ভাষা পছন্দ করি,আমি বাংলা ভাষা পছন্দ করি,Ami vAMlA bhAShA paChanda kari
9,Bengali,আজ আবহাওয়া ভালো,আজ আবহাওয়া ভালো,Aja AvahAoya়A bhAlo


In [23]:
# ============================================
# SCRIPT CHARACTER ANALYSIS
# ============================================

def script_statistics(samples):
    total_chars = sum(len(text.replace(" ", "")) for text in samples)
    unique_chars = set(
        char
        for text in samples
        for char in text
        if not char.isspace()
    )

    return total_chars, len(unique_chars)


hindi_chars, hindi_unique = script_statistics(normalized_hindi)
bengali_chars, bengali_unique = script_statistics(normalized_bengali)

script_stats = pd.DataFrame({
    "Language": ["Hindi", "Bengali"],
    "Total Characters": [hindi_chars, bengali_chars],
    "Unique Characters": [hindi_unique, bengali_unique]
})

script_stats

,Language,Total Characters,Unique Characters
0,Hindi,68,29
1,Bengali,70,29


In [24]:
# ============================================
# Q4(a): 20 HINDI-ENGLISH PARALLEL SENTENCES
# ============================================

parallel_data = {
    "Hindi": [
        "मेरा नाम राहुल है।",
        "मैं एक छात्र हूँ।",
        "मुझे कंप्यूटर विज्ञान पसंद है।",
        "भारत एक बड़ा देश है।",
        "आज मौसम अच्छा है।",
        "मैं रोज कॉलेज जाता हूँ।",
        "वह किताब पढ़ रही है।",
        "हम क्रिकेट खेलते हैं।",
        "मुझे हिंदी भाषा पसंद है।",
        "यह मेरा घर है।",
        "मेरी परीक्षा कल है।",
        "मैं Python सीख रहा हूँ।",
        "वह बाजार जा रहा है।",
        "आज रविवार है।",
        "मुझे संगीत सुनना पसंद है।",
        "दिल्ली भारत की राजधानी है।",
        "हम शाम को फिल्म देखेंगे।",
        "मुझे नई चीजें सीखना पसंद है।",
        "यह एक अच्छा विचार है।",
        "मैं अपने दोस्तों से मिला।"
    ],

    "English": [
        "My name is Rahul.",
        "I am a student.",
        "I like computer science.",
        "India is a large country.",
        "The weather is good today.",
        "I go to college every day.",
        "She is reading a book.",
        "We play cricket.",
        "I like the Hindi language.",
        "This is my house.",
        "My exam is tomorrow.",
        "I am learning Python.",
        "He is going to the market.",
        "Today is Sunday.",
        "I like listening to music.",
        "Delhi is the capital of India.",
        "We will watch a movie in the evening.",
        "I like learning new things.",
        "This is a good idea.",
        "I met my friends."
    ]
}

parallel_df = pd.DataFrame(parallel_data)

parallel_df.index = np.arange(1, len(parallel_df) + 1)
parallel_df.index.name = "Pair ID"

parallel_df

,Hindi,English
Pair ID,,
1,मेरा नाम राहुल है।,My name is Rahul.
2,मैं एक छात्र हूँ।,I am a student.
3,मुझे कंप्यूटर विज्ञान पसंद है।,I like computer science.
4,भारत एक बड़ा देश है।,India is a large country.
5,आज मौसम अच्छा है।,The weather is good today.
6,मैं रोज कॉलेज जाता हूँ।,I go to college every day.
7,वह किताब पढ़ रही है।,She is reading a book.
8,हम क्रिकेट खेलते हैं।,We play cricket.
9,मुझे हिंदी भाषा पसंद है।,I like the Hindi language.


In [25]:
# ============================================
# Q4(b): TOKENIZATION
# ============================================

def tokenize_text(text):
    text = unicodedata.normalize("NFC", text)
    return wordpunct_tokenize(text)


parallel_df["Hindi Tokens"] = parallel_df["Hindi"].apply(tokenize_text)
parallel_df["English Tokens"] = parallel_df["English"].apply(tokenize_text)

parallel_df[[
    "Hindi",
    "Hindi Tokens",
    "English",
    "English Tokens"
]]

,Hindi,Hindi Tokens,English,English Tokens
Pair ID,,,,
1,मेरा नाम राहुल है।,"[म, े, र, ा, न, ा, म, र, ा, ह, ु, ल, ह, ै।]",My name is Rahul.,"[My, name, is, Rahul, .]"
2,मैं एक छात्र हूँ।,"[म, ैं, एक, छ, ा, त, ्, र, ह, ूँ।]",I am a student.,"[I, am, a, student, .]"
3,मुझे कंप्यूटर विज्ञान पसंद है।,"[म, ु, झ, े, क, ं, प, ्, य, ू, टर, व, ि, ज, ्, ञ, ा, न, पस, ं, द, ह, ै।]",I like computer science.,"[I, like, computer, science, .]"
4,भारत एक बड़ा देश है।,"[भ, ा, रत, एक, बड, ़ा, द, े, श, ह, ै।]",India is a large country.,"[India, is, a, large, country, .]"
5,आज मौसम अच्छा है।,"[आज, म, ौ, सम, अच, ्, छ, ा, ह, ै।]",The weather is good today.,"[The, weather, is, good, today, .]"
6,मैं रोज कॉलेज जाता हूँ।,"[म, ैं, र, ो, ज, क, ॉ, ल, े, ज, ज, ा, त, ा, ह, ूँ।]",I go to college every day.,"[I, go, to, college, every, day, .]"
7,वह किताब पढ़ रही है।,"[वह, क, ि, त, ा, ब, पढ, ़, रह, ी, ह, ै।]",She is reading a book.,"[She, is, reading, a, book, .]"
8,हम क्रिकेट खेलते हैं।,"[हम, क, ्, र, ि, क, े, ट, ख, े, लत, े, ह, ैं।]",We play cricket.,"[We, play, cricket, .]"
9,मुझे हिंदी भाषा पसंद है।,"[म, ु, झ, े, ह, िं, द, ी, भ, ा, ष, ा, पस, ं, द, ह, ै।]",I like the Hindi language.,"[I, like, the, Hindi, language, .]"


In [26]:
# ============================================
# BASIC SENTENCE ALIGNMENT
# ============================================

for idx, row in parallel_df.iterrows():

    hindi_tokens = row["Hindi Tokens"]
    english_tokens = row["English Tokens"]

    print(f"\nPAIR {idx}")
    print("-" * 60)

    print("Hindi   :", row["Hindi"])
    print("English :", row["English"])

    print("Hindi tokens   :", hindi_tokens)
    print("English tokens :", english_tokens)


PAIR 1
------------------------------------------------------------
Hindi   : मेरा नाम राहुल है।
English : My name is Rahul.
Hindi tokens   : ['म', 'े', 'र', 'ा', 'न', 'ा', 'म', 'र', 'ा', 'ह', 'ु', 'ल', 'ह', 'ै।']
English tokens : ['My', 'name', 'is', 'Rahul', '.']

PAIR 2
------------------------------------------------------------
Hindi   : मैं एक छात्र हूँ।
English : I am a student.
Hindi tokens   : ['म', 'ैं', 'एक', 'छ', 'ा', 'त', '्', 'र', 'ह', 'ूँ।']
English tokens : ['I', 'am', 'a', 'student', '.']

PAIR 3
------------------------------------------------------------
Hindi   : मुझे कंप्यूटर विज्ञान पसंद है।
English : I like computer science.
Hindi tokens   : ['म', 'ु', 'झ', 'े', 'क', 'ं', 'प', '्', 'य', 'ू', 'टर', 'व', 'ि', 'ज', '्', 'ञ', 'ा', 'न', 'पस', 'ं', 'द', 'ह', 'ै।']
English tokens : ['I', 'like', 'computer', 'science', '.']

PAIR 4
------------------------------------------------------------
Hindi   : भारत एक बड़ा देश है।
English : India is a large country.
Hindi tokens

In [27]:
# ============================================
# Q4(c): SENTENCE LENGTH
# ============================================

parallel_df["Hindi Length"] = parallel_df["Hindi Tokens"].apply(len)
parallel_df["English Length"] = parallel_df["English Tokens"].apply(len)

length_stats = pd.DataFrame({
    "Language": ["Hindi", "English"],
    "Average Sentence Length": [
        parallel_df["Hindi Length"].mean(),
        parallel_df["English Length"].mean()
    ],
    "Maximum Sentence Length": [
        parallel_df["Hindi Length"].max(),
        parallel_df["English Length"].max()
    ],
    "Minimum Sentence Length": [
        parallel_df["Hindi Length"].min(),
        parallel_df["English Length"].min()
    ]
})

length_stats

,Language,Average Sentence Length,Maximum Sentence Length,Minimum Sentence Length
0,Hindi,13.95,23,8
1,English,5.75,9,4


In [28]:
# ============================================
# Q4(c): VOCABULARY SIZE
# ============================================

hindi_parallel_tokens = []

english_parallel_tokens = []

for tokens in parallel_df["Hindi Tokens"]:
    hindi_parallel_tokens.extend(tokens)

for tokens in parallel_df["English Tokens"]:
    english_parallel_tokens.extend(tokens)


hindi_vocab = set(hindi_parallel_tokens)
english_vocab = set(english_parallel_tokens)


parallel_vocab_stats = pd.DataFrame({
    "Language": ["Hindi", "English"],
    "Total Tokens": [
        len(hindi_parallel_tokens),
        len(english_parallel_tokens)
    ],
    "Vocabulary Size": [
        len(hindi_vocab),
        len(english_vocab)
    ]
})

parallel_vocab_stats

,Language,Total Tokens,Vocabulary Size
0,Hindi,279,71
1,English,115,60


In [29]:
# ============================================
# COMPLETE PARALLEL CORPUS SUMMARY
# ============================================

print("PARALLEL CORPUS STATISTICS")
print("=" * 50)

print("Number of sentence pairs:", len(parallel_df))

print("\nHindi:")
print("Total tokens:", len(hindi_parallel_tokens))
print("Vocabulary:", len(hindi_vocab))
print("Average sentence length:",
      round(parallel_df["Hindi Length"].mean(), 2))

print("\nEnglish:")
print("Total tokens:", len(english_parallel_tokens))
print("Vocabulary:", len(english_vocab))
print("Average sentence length:",
      round(parallel_df["English Length"].mean(), 2))

PARALLEL CORPUS STATISTICS
Number of sentence pairs: 20

Hindi:
Total tokens: 279
Vocabulary: 71
Average sentence length: 13.95

English:
Total tokens: 115
Vocabulary: 60
Average sentence length: 5.75


In [30]:
# ============================================
# Q4(d): 10 CODE-MIXED SENTENCES
# ============================================

code_mixed = [
    "आज मेरा college बहुत अच्छा था।",
    "मैंने Python का नया project बनाया।",
    "आज हमारी class बहुत interesting थी।",
    "मुझे यह assignment complete करना है।",
    "कल मेरा computer science का exam है।",
    "मैं आज gym के बाद coding करूंगा।",
    "यह movie बहुत interesting है।",
    "मुझे अपने project की presentation बनानी है।",
    "आज teacher ने नया topic explain किया।",
    "मैं weekend पर दोस्तों के साथ cricket खेलूंगा।"
]

print("Code-Mixed Sentences")
print("=" * 50)

for i, sentence in enumerate(code_mixed, 1):
    print(f"{i}. {sentence}")

Code-Mixed Sentences
1. आज मेरा college बहुत अच्छा था।
2. मैंने Python का नया project बनाया।
3. आज हमारी class बहुत interesting थी।
4. मुझे यह assignment complete करना है।
5. कल मेरा computer science का exam है।
6. मैं आज gym के बाद coding करूंगा।
7. यह movie बहुत interesting है।
8. मुझे अपने project की presentation बनानी है।
9. आज teacher ने नया topic explain किया।
10. मैं weekend पर दोस्तों के साथ cricket खेलूंगा।


In [31]:
# ============================================
# Q4(e): LANGUAGE SEGMENT IDENTIFICATION
# ============================================

def detect_token_language(token):

    if re.search(r'[\u0900-\u097F]', token):
        return "Hindi/Devanagari"

    elif re.search(r'[A-Za-z]', token):
        return "English/Roman"

    elif re.search(r'[\u0980-\u09FF]', token):
        return "Bengali"

    elif re.search(r'[\u0B80-\u0BFF]', token):
        return "Tamil"

    elif re.search(r'[\u0C00-\u0C7F]', token):
        return "Telugu"

    else:
        return "Other"


code_mixed_analysis = []

for sentence in code_mixed:

    tokens = wordpunct_tokenize(sentence)

    for token in tokens:

        if token.strip():
            code_mixed_analysis.append({
                "Sentence": sentence,
                "Token": token,
                "Language/Script": detect_token_language(token)
            })


code_mixed_df = pd.DataFrame(code_mixed_analysis)

code_mixed_df

,Sentence,Token,Language/Script
0,आज मेरा college बहुत अच्छा था।,आज,Hindi/Devanagari
1,आज मेरा college बहुत अच्छा था।,म,Hindi/Devanagari
2,आज मेरा college बहुत अच्छा था।,े,Hindi/Devanagari
3,आज मेरा college बहुत अच्छा था।,र,Hindi/Devanagari
4,आज मेरा college बहुत अच्छा था।,ा,Hindi/Devanagari
...,...,...,...
131,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।,े,Hindi/Devanagari
132,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।,ल,Hindi/Devanagari
133,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।,ूं,Hindi/Devanagari
134,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।,ग,Hindi/Devanagari


In [32]:
# ============================================
# SUMMARIZE EACH CODE-MIXED SENTENCE
# ============================================

summary = []

for i, sentence in enumerate(code_mixed, 1):

    tokens = [
        token for token in wordpunct_tokenize(sentence)
        if token.strip()
    ]

    hindi_tokens = [
        token for token in tokens
        if detect_token_language(token) == "Hindi/Devanagari"
    ]

    english_tokens = [
        token for token in tokens
        if detect_token_language(token) == "English/Roman"
    ]

    summary.append({
        "Sentence ID": i,
        "Original Sentence": sentence,
        "Total Tokens": len(tokens),
        "Indic Tokens": len(hindi_tokens),
        "English Tokens": len(english_tokens),
        "Indic Words": " ".join(hindi_tokens),
        "English Words": " ".join(english_tokens)
    })


code_mixed_summary = pd.DataFrame(summary)

code_mixed_summary

,Sentence ID,Original Sentence,Total Tokens,Indic Tokens,English Tokens,Indic Words,English Words
0,1,आज मेरा college बहुत अच्छा था।,15,14,1,आज म े र ा बह ु त अच ् छ ा थ ा।,college
1,2,मैंने Python का नया project बनाया।,14,12,2,म ैं न े क ा नय ा बन ा य ा।,Python project
2,3,आज हमारी class बहुत interesting थी।,12,10,2,आज हम ा र ी बह ु त थ ी।,class interesting
3,4,मुझे यह assignment complete करना है।,11,9,2,म ु झ े यह करन ा ह ै।,assignment complete
4,5,कल मेरा computer science का exam है।,12,9,3,कल म े र ा क ा ह ै।,computer science exam
5,6,मैं आज gym के बाद coding करूंगा।,14,12,2,म ैं आज क े ब ा द कर ूं ग ा।,gym coding
6,7,यह movie बहुत interesting है।,8,6,2,यह बह ु त ह ै।,movie interesting
7,8,मुझे अपने project की presentation बनानी है।,16,14,2,म ु झ े अपन े क ी बन ा न ी ह ै।,project presentation
8,9,आज teacher ने नया topic explain किया।,12,9,3,आज न े नय ा क ि य ा।,teacher topic explain
9,10,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।,22,20,2,म ैं पर द ो स ् त ों क े स ा थ ख े ल ूं ग ा।,weekend cricket


In [33]:
# ============================================
# NORMALIZE CODE-MIXED TEXT
# ============================================

code_mixed_normalized = [
    normalize_indic_text(sentence)
    for sentence in code_mixed
]

code_mixed_normalization_df = pd.DataFrame({
    "Original": code_mixed,
    "Normalized": code_mixed_normalized
})

code_mixed_normalization_df

,Original,Normalized
0,आज मेरा college बहुत अच्छा था।,आज मेरा college बहुत अच्छा था।
1,मैंने Python का नया project बनाया।,मैंने python का नया project बनाया।
2,आज हमारी class बहुत interesting थी।,आज हमारी class बहुत interesting थी।
3,मुझे यह assignment complete करना है।,मुझे यह assignment complete करना है।
4,कल मेरा computer science का exam है।,कल मेरा computer science का exam है।
5,मैं आज gym के बाद coding करूंगा।,मैं आज gym के बाद coding करूंगा।
6,यह movie बहुत interesting है।,यह movie बहुत interesting है।
7,मुझे अपने project की presentation बनानी है।,मुझे अपने project की presentation बनानी है।
8,आज teacher ने नया topic explain किया।,आज teacher ने नया topic explain किया।
9,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।,मैं weekend पर दोस्तों के साथ cricket खेलूंगा।


In [34]:
# ============================================
# OVERALL LANGUAGE DISTRIBUTION
# ============================================

language_counts = code_mixed_df["Language/Script"].value_counts()

print("Language/Script Token Distribution")
print("=" * 50)

display(
    language_counts.reset_index().rename(
        columns={
            "index": "Language/Script",
            "Language/Script": "Token Count"
        }
    )
)

Language/Script Token Distribution


,Token Count,count
0,Hindi/Devanagari,115
1,English/Roman,21


In [35]:
# ============================================
# FINAL LAB SUMMARY
# ============================================

final_summary = pd.DataFrame({
    "Component": [
        "Q1 - Indic Corpus",
        "Q2 - Noisy Text",
        "Q3 - Hindi Script",
        "Q3 - Bengali Script",
        "Q4 - Parallel Corpus",
        "Q4 - Code-Mixed Text"
    ],
    "Samples / Pairs": [
        len(plain_hindi_sentences),
        len(noisy_texts),
        len(hindi_samples),
        len(bengali_samples),
        len(parallel_df),
        len(code_mixed)
    ],
    "Task": [
        "NLTK corpus analysis",
        "Text normalization",
        "Script normalization/transliteration",
        "Script normalization/transliteration",
        "Hindi-English analysis",
        "Language identification + normalization"
    ]
})

final_summary

,Component,Samples / Pairs,Task
0,Q1 - Indic Corpus,500,NLTK corpus analysis
1,Q2 - Noisy Text,20,Text normalization
2,Q3 - Hindi Script,5,Script normalization/transliteration
3,Q3 - Bengali Script,5,Script normalization/transliteration
4,Q4 - Parallel Corpus,20,Hindi-English analysis
5,Q4 - Code-Mixed Text,10,Language identification + normalization


In [36]:
# ============================================
# SAVE OUTPUT DATASETS
# ============================================

normalization_df.to_csv(
    "/kaggle/working/normalized_indic_text.csv",
    index=False,
    encoding="utf-8-sig"
)

script_comparison.to_csv(
    "/kaggle/working/script_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

parallel_df.to_csv(
    "/kaggle/working/parallel_corpus.csv",
    index=False,
    encoding="utf-8-sig"
)

code_mixed_summary.to_csv(
    "/kaggle/working/code_mixed_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV files saved successfully!")

CSV files saved successfully!


In [37]:
# ============================================
# SAVE ALL RESULTS TO ONE EXCEL FILE
# ============================================

with pd.ExcelWriter(
    "/kaggle/working/NLP_Lab_4_Results.xlsx",
    engine="openpyxl"
) as writer:

    stats_df.to_excel(
        writer,
        sheet_name="Q1_Statistics",
        index=False
    )

    normalization_df.to_excel(
        writer,
        sheet_name="Q2_Normalization",
        index=False
    )

    script_comparison.to_excel(
        writer,
        sheet_name="Q3_Scripts",
        index=False
    )

    parallel_df.to_excel(
        writer,
        sheet_name="Q4_Parallel",
        index=False
    )

    code_mixed_summary.to_excel(
        writer,
        sheet_name="Q4_CodeMixed",
        index=False
    )

print("Excel file created:")
print("/kaggle/working/NLP_Lab_4_Results.xlsx")

Excel file created:
/kaggle/working/NLP_Lab_4_Results.xlsx
